In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
from joblib import Parallel, delayed
from skimage.color import rgb2gray
from skimage.feature import hog
from skimage.io import imread
from skimage.transform import resize
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

import joblib
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

e:\CS-UIT\Ki 6\Nhập môn CV\CS231.Q22-TrafficSign-Classification\cs231-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def resolve_data_dir() -> Path:
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        candidate = parent / "data"
        if (candidate / "train").is_dir() and (candidate / "test").is_dir():
            return candidate
    raise FileNotFoundError("Could not find data/train and data/test")

def collect_paths(split_dir: Path) -> tuple[list[Path], np.ndarray, list[str]]:
    labels = sorted([d.name for d in split_dir.iterdir() if d.is_dir()])
    if not labels:
        raise RuntimeError(f"No class folders found in {split_dir}")

    paths: list[Path] = []
    y: list[str] = []
    for label in labels:
        for path in sorted((split_dir / label).glob("*.*")):
            if path.is_file():
                paths.append(path)
                y.append(label)

    if not paths:
        raise RuntimeError(f"No images found in {split_dir}")

    return paths, np.array(y), labels

def compute_features(
    paths: list[Path],
    labels: np.ndarray,
    image_size: tuple[int, int],
    hog_params: dict,
    cache_file: Path | None,
    n_jobs: int,
) -> tuple[np.ndarray, np.ndarray]:
    if cache_file is not None and cache_file.exists():
        data = np.load(cache_file)
        return data["X"], data["y"]

    def extract(path: Path) -> np.ndarray | None:
        try:
            img = imread(path)
            gray = rgb2gray(img) if img.ndim == 3 else img
            gray = resize(gray, image_size, anti_aliasing=True)
            return hog(gray, **hog_params)
        except Exception:
            return None

    desc = f"HOG {cache_file.stem}" if cache_file is not None else "HOG"
    features = Parallel(n_jobs=n_jobs)(
        delayed(extract)(p) for p in tqdm(paths, desc=desc, total=len(paths))
    )

    kept_feats: list[np.ndarray] = []
    kept_labels: list[str] = []
    skipped = 0
    for feat, label in zip(features, labels):
        if feat is None:
            skipped += 1
            continue
        kept_feats.append(feat)
        kept_labels.append(label)

    if not kept_feats:
        raise RuntimeError("No features extracted; check input images.")

    X = np.vstack(kept_feats)
    y = np.array(kept_labels)

    if cache_file is not None:
        cache_file.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(cache_file, X=X, y=y)

    if skipped:
        print(f"Skipped {skipped} unreadable files.")

    return X, y

def objective(trial: optuna.Trial) -> float:
    params = {
        "n_estimators": 10,
        "max_depth": trial.suggest_int("max_depth", 5, 50),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "random_state": SEED,
        "warm_start": True,
        "n_jobs": 1,
    }
    clf = RandomForestClassifier(**params)
    acc = 0.0
    for n_estimators in ESTIMATOR_STEPS:
        clf.set_params(n_estimators=n_estimators)
        clf.fit(X_train, y_train_enc)
        acc = accuracy_score(y_val_enc, clf.predict(X_val))
        trial.report(acc, n_estimators)
        if trial.should_prune():
            raise optuna.TrialPruned()
    trial.set_user_attr("model", clf)
    return acc

In [3]:
SEED = 42
IMAGE_SIZE = (128, 128)
HOG_PARAMS = {
    "orientations": 9,
    "pixels_per_cell": (8, 8),
    "cells_per_block": (2, 2),
    "block_norm": "L2-Hys",
}
VAL_SIZE = 0.2
N_JOBS = 8
OPTUNA_TRIALS = 50
OPTUNA_JOBS = 4
RF_JOBS = 4
ESTIMATOR_STEPS = [50, 100, 150, 200]

DATA_DIR = resolve_data_dir()
MODELS_DIR = DATA_DIR.parent / "models"
CACHE_DIR = DATA_DIR.parent / "cache"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
train_paths, train_labels, train_classes = collect_paths(DATA_DIR / "train")
test_paths, test_labels, test_classes = collect_paths(DATA_DIR / "test")

if set(train_classes) != set(test_classes):
    print("Warning: train/test class folders differ; using union of labels.")
all_classes = sorted(set(train_classes) | set(test_classes))

train_paths, val_paths, y_train, y_val = train_test_split(
    train_paths,
    train_labels,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=train_labels,
)

cache_tag = (
    f"hog_rf_8x2_{IMAGE_SIZE[0]}x{IMAGE_SIZE[1]}_"
    f"val{int(VAL_SIZE * 100)}_seed{SEED}"
)
X_train, y_train = compute_features(
    train_paths, y_train, IMAGE_SIZE, HOG_PARAMS, CACHE_DIR / f"{cache_tag}_train.npz", N_JOBS
)
X_val, y_val = compute_features(
    val_paths, y_val, IMAGE_SIZE, HOG_PARAMS, CACHE_DIR / f"{cache_tag}_val.npz", N_JOBS
)
X_test, y_test = compute_features(
    test_paths, test_labels, IMAGE_SIZE, HOG_PARAMS, CACHE_DIR / f"{cache_tag}_test.npz", N_JOBS
)

le = LabelEncoder()
le.fit(all_classes)
y_train_enc = le.transform(y_train)
y_val_enc = le.transform(y_val)
y_test_enc = le.transform(y_test)

HOG hog_rf_8x2_128x128_val20_seed42_train: 100%|██████████| 1075/1075 [00:04<00:00, 230.45it/s]


Skipped 116 unreadable files.


HOG hog_rf_8x2_128x128_val20_seed42_val: 100%|██████████| 269/269 [00:00<00:00, 684.25it/s]


Skipped 32 unreadable files.


HOG hog_rf_8x2_128x128_val20_seed42_test: 100%|██████████| 155/155 [00:00<00:00, 655.59it/s]


In [5]:
study = optuna.create_study(
    direction="maximize",
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=1, interval_steps=1),
    sampler=TPESampler(multivariate=True),
)

e:\CS-UIT\Ki 6\Nhập môn CV\CS231.Q22-TrafficSign-Classification\cs231-venv\Lib\site-packages\optuna\samplers\_tpe\sampler.py:319: ExperimentalWarning: ``multivariate`` option is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2026-05-12 23:06:11,074] A new study created in memory with name: no-name-cca443be-9d47-45d3-82a6-3910e873c2ad


In [6]:
study.optimize(objective, n_trials=OPTUNA_TRIALS, n_jobs=OPTUNA_JOBS)

print("Best validation accuracy:", study.best_value)
print("Best params:", study.best_params)

[I 2026-05-12 23:06:13,649] Trial 1 finished with value: 0.7215189873417721 and parameters: {'max_depth': 49, 'min_samples_split': 16, 'min_samples_leaf': 5, 'max_features': 'log2'}. Best is trial 1 with value: 0.7215189873417721.
[I 2026-05-12 23:06:13,812] Trial 3 finished with value: 0.729957805907173 and parameters: {'max_depth': 33, 'min_samples_split': 15, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 3 with value: 0.729957805907173.
[I 2026-05-12 23:06:25,971] Trial 0 finished with value: 0.729957805907173 and parameters: {'max_depth': 8, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 3 with value: 0.729957805907173.
[I 2026-05-12 23:06:29,482] Trial 5 finished with value: 0.7215189873417721 and parameters: {'max_depth': 32, 'min_samples_split': 17, 'min_samples_leaf': 7, 'max_features': 'sqrt'}. Best is trial 3 with value: 0.729957805907173.
[I 2026-05-12 23:06:29,787] Trial 2 finished with value: 0.759493670886076 and para

Best validation accuracy: 0.7890295358649789
Best params: {'max_depth': 37, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': 'sqrt'}


In [7]:
X_combined = np.vstack([X_train, X_val])
y_combined = np.concatenate([y_train_enc, y_val_enc])

best_params = dict(study.best_params)
best_params.update(
    {
        "n_estimators": max(ESTIMATOR_STEPS),
        "random_state": SEED,
        "warm_start": False,
        "n_jobs": RF_JOBS,
    }
)

best_clf = RandomForestClassifier(**best_params)
best_clf.fit(X_combined, y_combined)

y_pred_test = best_clf.predict(X_test)
print("\nTest Accuracy:", accuracy_score(y_test_enc, y_pred_test))
print("\nClassification Report:")
print(classification_report(y_test_enc, y_pred_test, target_names=le.classes_))


Test Accuracy: 0.7225806451612903

Classification Report:
              precision    recall  f1-score   support

         Cam       0.77      0.94      0.85        36
      Chidan       0.59      0.61      0.60        36
    Hieulenh       0.72      0.42      0.53        31
    Nguyhiem       0.90      0.97      0.93        29
         Phu       0.60      0.65      0.62        23

    accuracy                           0.72       155
   macro avg       0.72      0.72      0.71       155
weighted avg       0.72      0.72      0.71       155



In [8]:
model_path = MODELS_DIR / "HOG_RandomForest_8x2.joblib"
payload = {
    "model": best_clf,
    "label_encoder": le,
    "hog_params": HOG_PARAMS,
    "image_size": IMAGE_SIZE,
    "classes": list(le.classes_),
}
joblib.dump(payload, model_path)
print(f"Saved model to {model_path}")

Saved model to E:\CS-UIT\Ki 6\Nhập môn CV\CS231.Q22-TrafficSign-Classification\models\HOG_RandomForest_8x2.joblib
